# SupCon teacher: WideResNet-22 (4,999,434 params) trained with Supervised Contrastive Learning

Khosla et al., NeurIPS 2020 (arXiv 2004.11362), settings from the paper and the official repo (github.com/HobbitLong/SupContrast):
- **Stage 1** (350 epochs): no classifier. Two augmented views per image; same-class images are pulled together in a 128-d embedding, other classes pushed apart. SupCon loss, temperature 0.1, batch 1024, SGD LR 0.5, cosine.
- **Stage 2** (50 epochs): freeze the network, train only the final linear classifier with cross-entropy (batch 512, LR 5).

Purpose: a **teacher** for the next distillation run (trained with a different loss, so it may make different mistakes). `best.pt` is saved in the same format as our other runs, so the distillation notebook's section 9b picks it up automatically.

Our choices (not in the paper): WideResNet-22 with stage widths 68/132/280 (4,999,434 params, just under the 5M limit; standard WRN-22-4 is 64/128/256 = 4,298,970) instead of ResNet-50; our augmentation (crop, flip, TrivialAugment, grayscale, cutout); no mixing, no SLA, no ASAM.

A100. Run all; if Colab disconnects, Run all again: both stages resume.

## 1. Config

In [1]:
CFG = dict(
    RUN_NAME="supcon_wrn22_5M_350ep_v1",
    DRIVE_PROJECT_DIR="cifar10_resnet_challenge",
    SPLIT_FILE="split_seed42_val0.05_test0.05_n50000.npz",
    SEED=31,

    # model: WideResNet-22, stage widths 68/132/280 = 4,999,434 params (WRN-22-4 widened to just under 5M), no dropout
    DEPTH=22, WIDEN=(68, 132, 280),

    # stage 1: SupCon (paper + official repo)
    S1_EPOCHS=350, S1_BATCH=1024,      # 1024 images = 2048 views per step (official repo setting)
     S1_LR=0.5, S1_WARMUP=10, S1_WD=1e-4, TEMP=0.1, BASE_TEMP=0.07, PROJ_DIM=128,

    # stage 2: linear classifier on the frozen network (official repo: batch 512, LR 5)
    S2_EPOCHS=50, S2_BATCH=512, S2_LR=5.0,

    # augmentation (ours)
    TA_GROUP=16, GRAYSCALE_P=0.1, CUTOUT=8,
)
CFG

{'RUN_NAME': 'supcon_wrn22_5M_350ep_v1',
 'DRIVE_PROJECT_DIR': 'cifar10_resnet_challenge',
 'SPLIT_FILE': 'split_seed42_val0.05_test0.05_n50000.npz',
 'SEED': 31,
 'DEPTH': 22,
 'WIDEN': (68, 132, 280),
 'S1_EPOCHS': 350,
 'S1_BATCH': 1024,
 'S1_LR': 0.5,
 'S1_WARMUP': 10,
 'S1_WD': 0.0001,
 'TEMP': 0.1,
 'BASE_TEMP': 0.07,
 'PROJ_DIM': 128,
 'S2_EPOCHS': 50,
 'S2_BATCH': 512,
 'S2_LR': 5.0,
 'TA_GROUP': 16,
 'GRAYSCALE_P': 0.1,
 'CUTOUT': 8}

## 2. Setup

In [2]:
import os, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.transforms import v2 as T
from google.colab import drive

drive.mount("/content/drive")
ROOT = os.path.join("/content/drive/MyDrive", CFG["DRIVE_PROJECT_DIR"])
CACHE = os.path.join(ROOT, "data", "cache")
RUN_DIR = os.path.join(ROOT, "runs", CFG["RUN_NAME"]); os.makedirs(RUN_DIR, exist_ok=True)
S1_LAST, S2_LAST, BEST = (os.path.join(RUN_DIR, f) for f in ("stage1_last.pt", "stage2_last.pt", "best.pt"))

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
torch.manual_seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
print("GPU:", torch.cuda.get_device_name(0), "| run dir:", RUN_DIR)

def save(obj, path):
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB | run dir: /content/drive/MyDrive/cifar10_resnet_challenge/runs/supcon_wrn22_5M_350ep_v1


## 3. Data (same 45k / 2.5k / 2.5k split as every run)

In [3]:
X = np.load(os.path.join(CACHE, "train_x.npy")); y = np.load(os.path.join(CACHE, "train_y.npy"))
split = np.load(os.path.join(ROOT, "data", "splits", CFG["SPLIT_FILE"]))
def gpu(a): return torch.from_numpy(a).permute(0, 3, 1, 2).contiguous().to(device)
Xtr, ytr = gpu(X[split["train"]]), torch.from_numpy(y[split["train"]]).to(device)
Xva, yva = gpu(X[split["val"]]), torch.from_numpy(y[split["val"]]).to(device)
Xte, yte = gpu(X[split["test"]]), torch.from_numpy(y[split["test"]]).to(device)

_f = Xtr.float() / 255                                   # same statistics as every other run
MEAN = _f.mean(dim=(0, 2, 3)).view(1, 3, 1, 1); STD = _f.std(dim=(0, 2, 3)).view(1, 3, 1, 1); del _f
def normalize(x): return ((x.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)
print("train", len(Xtr), "| val", len(Xva), "| internal test", len(Xte))

train 45000 | val 2500 | internal test 2500


## 4. Augmentation (no mixing: SupCon needs each view to belong to one class)

In [4]:
trivial = T.TrivialAugmentWide()

def augment(x_u8):
    n, ar = len(x_u8), torch.arange(32, device=device)
    x_u8 = torch.cat([trivial(c) for c in x_u8.split(CFG["TA_GROUP"])])
    gray = (x_u8.float() * torch.tensor([0.299, 0.587, 0.114], device=device).view(1, 3, 1, 1)).sum(1, keepdim=True)
    g = torch.rand(n, device=device) < CFG["GRAYSCALE_P"]
    x_u8 = torch.where(g[:, None, None, None], gray.round().to(torch.uint8).expand(-1, 3, -1, -1), x_u8)
    x = (x_u8.float() / 255 - MEAN) / STD
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    dy, dx = torch.randint(0, 9, (n,), device=device), torch.randint(0, 9, (n,), device=device)
    x = xp[torch.arange(n, device=device)[:, None, None, None], torch.arange(3, device=device)[None, :, None, None],
           (dy[:, None] + ar)[:, None, :, None], (dx[:, None] + ar)[:, None, None, :]]
    flip = torch.rand(n, device=device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    cy, cx, h = torch.randint(0, 32, (n,), device=device), torch.randint(0, 32, (n,), device=device), CFG["CUTOUT"] // 2
    hole = ((ar[None, :, None] - cy[:, None, None]).abs() < h) & ((ar[None, None, :] - cx[:, None, None]).abs() < h)
    return (x * (~hole)[:, None].float()).contiguous(memory_format=torch.channels_last)

def crop_flip(x_u8):
    """Light augmentation for stage 2 (the official repo also uses crop + flip for the linear classifier)."""
    n, ar = len(x_u8), torch.arange(32, device=device)
    x = F.pad((x_u8.float() / 255 - MEAN) / STD, (4, 4, 4, 4), mode="reflect")
    dy, dx = torch.randint(0, 9, (n,), device=device), torch.randint(0, 9, (n,), device=device)
    x = x[torch.arange(n, device=device)[:, None, None, None], torch.arange(3, device=device)[None, :, None, None],
          (dy[:, None] + ar)[:, None, :, None], (dx[:, None] + ar)[:, None, None, :]]
    flip = torch.rand(n, device=device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x).contiguous(memory_format=torch.channels_last)

## 5. Model: WideResNet-22, widths 68/132/280 (same class as in the distillation notebook, so it loads as a teacher) + projection head

In [5]:
class WBlock(nn.Module):
    """Pre-activation wide residual block (Zagoruyko & Komodakis 2016)."""
    def __init__(self, cin, cout, stride, drop):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(cin), nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(cout), nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.dropout = nn.Dropout(drop)
        self.skip = None if (stride == 1 and cin == cout) else nn.Conv2d(cin, cout, 1, stride, bias=False)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.skip is None else self.skip(o)
        return self.conv2(self.dropout(F.relu(self.bn2(self.conv1(o))))) + sc

class WRN(nn.Module):
    """WideResNet for CIFAR (Zagoruyko & Komodakis 2016). k = widen factor (WRN-28-10: 36.5M params) or explicit stage widths."""
    def __init__(self, depth=28, k=10, dropout=0.0, num_classes=10):
        super().__init__()
        n = (depth - 4) // 6
        w = [16] + (list(k) if isinstance(k, (tuple, list)) else [16 * k, 32 * k, 64 * k])   # k: widen factor, or the 3 stage widths
        self.stem = nn.Conv2d(3, w[0], 3, 1, 1, bias=False)
        layers, cin = [], w[0]
        for i in range(3):
            for j in range(n):
                layers.append(WBlock(cin, w[i + 1], 2 if (j == 0 and i > 0) else 1, dropout)); cin = w[i + 1]
        self.layers = nn.Sequential(*layers); self.bn = nn.BatchNorm2d(cin)
        self.drop = nn.Dropout(0.0); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(F.relu(self.bn(self.layers(self.stem(x)))), 1), 1)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))

model = WRN(CFG["DEPTH"], CFG["WIDEN"]).to(device).to(memory_format=torch.channels_last)
dim = model.fc.in_features
assert sum(p.numel() for p in model.parameters()) <= 5_000_000
head = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(inplace=True), nn.Linear(dim, CFG["PROJ_DIM"])).to(device)   # thrown away after stage 1
print(f"WRN-{CFG['DEPTH']} widths {CFG['WIDEN']} parameters (incl. classifier): {sum(p.numel() for p in model.parameters()):,} | projection head (training only): {sum(p.numel() for p in head.parameters()):,}")

@torch.no_grad()
def feats(X, bs=1000):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out.append(model.features(normalize(X[i:i + bs])).float())
    return torch.cat(out)

WRN-22 widths (68, 132, 280) parameters (incl. classifier): 4,999,434 | projection head (training only): 114,648


## 6. Stage 1: SupCon pre-training (resumes from stage1_last.pt)

In [6]:
def supcon_loss(z, labels):
    """Supervised contrastive loss (paper eq. 2, 'L_out'). z: (2B, d) L2-normalised, labels: (2B,)."""
    sim = z @ z.T / CFG["TEMP"]
    self_mask = torch.eye(len(z), device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(self_mask, -1e9)
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos = (labels[:, None] == labels[None, :]) & ~self_mask
    mean_log_pos = (log_prob * pos).sum(1) / pos.sum(1).clamp_min(1)
    return -(CFG["TEMP"] / CFG["BASE_TEMP"]) * mean_log_pos.mean()          # scaling as in the official code

params1 = list(model.parameters()) + list(head.parameters())
decay = [p for p in params1 if p.ndim > 1]; no_decay = [p for p in params1 if p.ndim <= 1]
opt1 = torch.optim.SGD([{"params": decay, "weight_decay": CFG["S1_WD"]}, {"params": no_decay, "weight_decay": 0.0}],
                       lr=CFG["S1_LR"], momentum=0.9)
steps1 = len(Xtr) // CFG["S1_BATCH"]; warm1 = CFG["S1_WARMUP"] * steps1
sched1 = torch.optim.lr_scheduler.SequentialLR(opt1, [
    torch.optim.lr_scheduler.LinearLR(opt1, start_factor=0.01, total_iters=warm1),
    torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=CFG["S1_EPOCHS"] * steps1 - warm1)], milestones=[warm1])

def proto_acc():
    """Progress check without a classifier: nearest class-mean (cosine) accuracy on validation."""
    ft, fv = F.normalize(feats(Xtr), dim=1), F.normalize(feats(Xva), dim=1)
    protos = F.normalize(torch.stack([ft[ytr == c].mean(0) for c in range(10)]), dim=1)
    return ((fv @ protos.T).argmax(1) == yva).float().mean().item()

start1, hist1 = 0, []
if os.path.exists(S1_LAST):
    ck = torch.load(S1_LAST, map_location=device, weights_only=False)
    assert ck["cfg"] == CFG, "Config changed since this run started: use a new RUN_NAME"
    model.load_state_dict(ck["model"]); head.load_state_dict(ck["head"]); opt1.load_state_dict(ck["opt"]); sched1.load_state_dict(ck["sched"])
    start1, hist1 = ck["epoch"], ck["history"]
    print(f"stage 1 resumed at epoch {start1}")

for epoch in range(start1, CFG["S1_EPOCHS"]):
    t0 = time.time(); model.train(); head.train(); tot = torch.zeros((), device=device)
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(steps1):                                   # drop the last partial batch
        idx = perm[i * CFG["S1_BATCH"]:(i + 1) * CFG["S1_BATCH"]]
        xb = Xtr[idx]
        views = torch.cat([augment(xb), augment(xb)])          # two independent views of every image
        labels = ytr[idx].repeat(2)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            z = head(model.features(views))
        loss = supcon_loss(F.normalize(z.float(), dim=1), labels)
        opt1.zero_grad(set_to_none=True); loss.backward(); opt1.step(); sched1.step()
        tot += loss.detach()
    rec = {"epoch": epoch + 1, "loss": tot.item() / steps1, "lr": sched1.get_last_lr()[0], "sec": time.time() - t0}
    if (epoch + 1) % 25 == 0 or epoch + 1 == CFG["S1_EPOCHS"]:
        rec["val_proto_acc"] = proto_acc()
    hist1.append(rec)
    save({"model": model.state_dict(), "head": head.state_dict(), "opt": opt1.state_dict(), "sched": sched1.state_dict(),
          "epoch": epoch + 1, "history": hist1, "cfg": CFG}, S1_LAST)
    print(f"stage 1 ep {epoch+1:3d}/{CFG['S1_EPOCHS']} | SupCon loss {rec['loss']:.4f} | lr {rec['lr']:.4f} | {rec['sec']:.0f}s"
          + (f" | val nearest-class-mean acc {rec['val_proto_acc']:.4f}" if "val_proto_acc" in rec else "")
          + f" | eta {(CFG['S1_EPOCHS']-epoch-1)*rec['sec']/3600:.1f} h")
print("stage 1 done")

stage 1 ep   1/350 | SupCon loss 10.8469 | lr 0.0545 | 26s | eta 2.5 h
stage 1 ep   2/350 | SupCon loss 10.7164 | lr 0.1040 | 10s | eta 1.0 h
stage 1 ep   3/350 | SupCon loss 10.6477 | lr 0.1535 | 10s | eta 1.0 h
stage 1 ep   4/350 | SupCon loss 10.6006 | lr 0.2030 | 10s | eta 1.0 h
stage 1 ep   5/350 | SupCon loss 10.4557 | lr 0.2525 | 10s | eta 1.0 h
stage 1 ep   6/350 | SupCon loss 10.3979 | lr 0.3020 | 11s | eta 1.0 h
stage 1 ep   7/350 | SupCon loss 10.3286 | lr 0.3515 | 10s | eta 1.0 h
stage 1 ep   8/350 | SupCon loss 10.2233 | lr 0.4010 | 10s | eta 1.0 h
stage 1 ep   9/350 | SupCon loss 10.1290 | lr 0.4505 | 10s | eta 1.0 h
stage 1 ep  10/350 | SupCon loss 10.0482 | lr 0.5000 | 10s | eta 1.0 h
stage 1 ep  11/350 | SupCon loss 9.9050 | lr 0.5000 | 10s | eta 1.0 h
stage 1 ep  12/350 | SupCon loss 9.7579 | lr 0.5000 | 10s | eta 1.0 h
stage 1 ep  13/350 | SupCon loss 9.7127 | lr 0.4999 | 10s | eta 1.0 h
stage 1 ep  14/350 | SupCon loss 9.6354 | lr 0.4998 | 10s | eta 1.0 h
stage 1 ep

## 7. Stage 2: linear classifier on the frozen network (resumes from stage2_last.pt)

In [7]:
for p in model.parameters(): p.requires_grad_(False)
model.fc = nn.Linear(model.fc.in_features, 10).to(device)          # fresh classifier; the only trained part
opt2 = torch.optim.SGD(model.fc.parameters(), lr=CFG["S2_LR"], momentum=0.9, weight_decay=0.0)
steps2 = math.ceil(len(Xtr) / CFG["S2_BATCH"])
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=CFG["S2_EPOCHS"] * steps2)

@torch.no_grad()
def predict(X, tta=True, bs=1000):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        with torch.autocast("cuda", dtype=torch.bfloat16):
            p = model(xb).float().softmax(1)
            if tta: p = (p + model(xb.flip(3)).float().softmax(1)) / 2
        out.append(p)
    return torch.cat(out)
def acc(p, yy): return (p.argmax(1) == yy).float().mean().item()

TEACHER_CFG = dict(ARCH="wrn", DEPTH=CFG["DEPTH"], WIDEN=CFG["WIDEN"], DROPOUT=0.0, SLA=False,
                   SPLIT_SEED=42, VAL_FRAC=0.05, TEST_FRAC=0.05, RUN_NAME=CFG["RUN_NAME"])

start2, best_acc, hist2 = 0, 0.0, []
if os.path.exists(S2_LAST):
    ck = torch.load(S2_LAST, map_location=device, weights_only=False)
    model.fc.load_state_dict(ck["fc"]); opt2.load_state_dict(ck["opt"]); sched2.load_state_dict(ck["sched"])
    start2, best_acc, hist2 = ck["epoch"], ck["best_acc"], ck["history"]
    print(f"stage 2 resumed at epoch {start2}")

for epoch in range(start2, CFG["S2_EPOCHS"]):
    t0 = time.time(); model.eval(); model.fc.train()             # network frozen (BatchNorm statistics too)
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), CFG["S2_BATCH"]):
        idx = perm[i:i + CFG["S2_BATCH"]]
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            f = model.features(crop_flip(Xtr[idx])).float()
        loss = F.cross_entropy(model.fc(f), ytr[idx])
        opt2.zero_grad(set_to_none=True); loss.backward(); opt2.step(); sched2.step()
    val_acc = acc(predict(Xva, tta=False), yva)
    hist2.append({"epoch": epoch + 1, "val_acc": val_acc})
    if val_acc > best_acc:
        best_acc = val_acc
        save({"model": model.state_dict(), "cfg": TEACHER_CFG, "epoch": epoch + 1, "val_acc": val_acc,
              "n_params": sum(p.numel() for p in model.parameters()), "norm": {"mean": MEAN.cpu(), "std": STD.cpu()},
              "is_ema": False}, BEST)
    save({"fc": model.fc.state_dict(), "opt": opt2.state_dict(), "sched": sched2.state_dict(),
          "epoch": epoch + 1, "best_acc": best_acc, "history": hist2}, S2_LAST)
    print(f"stage 2 ep {epoch+1:2d}/{CFG['S2_EPOCHS']} | loss {loss.item():.4f} | val acc {val_acc:.4f}{' *' if best_acc == val_acc else '  '} | {time.time()-t0:.0f}s")
print("stage 2 done, best val", best_acc)

stage 2 ep  1/50 | loss 0.0000 | val acc 0.9636 * | 3s
stage 2 ep  2/50 | loss 0.0000 | val acc 0.9640 * | 2s
stage 2 ep  3/50 | loss 0.0000 | val acc 0.9660 * | 2s
stage 2 ep  4/50 | loss 0.0000 | val acc 0.9652   | 1s
stage 2 ep  5/50 | loss 0.0000 | val acc 0.9648   | 1s
stage 2 ep  6/50 | loss 0.0000 | val acc 0.9664 * | 2s
stage 2 ep  7/50 | loss 0.0000 | val acc 0.9668 * | 2s
stage 2 ep  8/50 | loss 0.0000 | val acc 0.9668 * | 1s
stage 2 ep  9/50 | loss 0.0000 | val acc 0.9672 * | 2s
stage 2 ep 10/50 | loss 0.0000 | val acc 0.9672 * | 1s
stage 2 ep 11/50 | loss 0.0000 | val acc 0.9676 * | 2s
stage 2 ep 12/50 | loss 0.0000 | val acc 0.9676 * | 1s
stage 2 ep 13/50 | loss 0.0000 | val acc 0.9676 * | 1s
stage 2 ep 14/50 | loss 0.0000 | val acc 0.9680 * | 2s
stage 2 ep 15/50 | loss 0.0000 | val acc 0.9684 * | 2s
stage 2 ep 16/50 | loss 0.0000 | val acc 0.9680   | 1s
stage 2 ep 17/50 | loss 0.0000 | val acc 0.9684 * | 1s
stage 2 ep 18/50 | loss 0.0000 | val acc 0.9688 * | 2s
stage 2 ep

## 8. Evaluate best.pt

In [9]:
b = torch.load(BEST, map_location=device, weights_only=False)
model.load_state_dict(b["model"]); model.eval()
p_va, p_te = predict(Xva), predict(Xte)
print(f"best.pt: stage-2 epoch {b['epoch']} | VAL {acc(p_va, yva):.4f} | INTERNAL TEST {acc(p_te, yte):.4f}  (flip TTA)")
cm = torch.bincount(yte * 10 + p_te.argmax(1), minlength=100).view(10, 10).cpu()
print(pd.DataFrame({"recall": (cm.diag() / cm.sum(1)).numpy().round(3), "errors": (cm.sum(1) - cm.diag()).numpy()}, index=CLASSES).T.to_string())
off = cm.clone(); off.fill_diagonal_(0)
top = sorted(((int(off[i, j]), CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if off[i, j] > 0), reverse=True)[:6]
print("top confusions (true -> predicted):", ", ".join(f"{n} {a}->{c}" for n, a, c in top))

# curves: stage-1 loss and nearest-class-mean accuracy, stage-2 validation accuracy
h1, h2 = pd.DataFrame(hist1), pd.DataFrame(hist2)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(h1["epoch"], h1["loss"]); ax[0].set_title("stage 1: SupCon loss"); ax[0].set_xlabel("epoch")
if "val_proto_acc" in h1:
    d = h1.dropna(subset=["val_proto_acc"]); ax[1].plot(d["epoch"], d["val_proto_acc"], marker="o")
ax[1].set_title("stage 1: val nearest-class-mean accuracy"); ax[1].set_xlabel("epoch")
ax[2].plot(h2["epoch"], h2["val_acc"]); ax[2].set_title("stage 2: val accuracy (linear classifier)"); ax[2].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "curves.png"), dpi=110); plt.show()

# confusion matrix heatmap, internal test
fig, ax = plt.subplots(figsize=(7.5, 6.5))
cmn = cm.float() / cm.sum(1, keepdim=True).float()
im = ax.imshow(cmn.numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=45, ha="right"); ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Internal test, row-normalised (counts shown), acc {acc(p_te, yte):.4f}")
plt.colorbar(im); plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "confusion_internal_test.png"), dpi=120); plt.show()
print("\nThe distillation notebook ignores teachers with val below KD_MIN_TEACHER_VAL (0.955 in the generation-2 notebook). If VAL is below that, lower it there to include this model.")

best.pt: stage-2 epoch 31 | VAL 0.9772 | INTERNAL TEST 0.9688  (flip TTA)
        airplane  automobile    bird     cat   deer    dog  frog  horse   ship  truck
recall      0.96       0.984   0.936   0.936  0.972   0.94  0.98  0.988  0.992    1.0
errors     10.00       4.000  16.000  16.000  7.000  15.00  5.00  3.000  2.000    0.0
top confusions (true -> predicted): 14 cat->dog, 11 dog->cat, 7 bird->cat, 4 automobile->truck, 3 dog->horse, 3 deer->horse


NameError: name 'plt' is not defined

In [ ]:
# ---- 9. Kaggle submission from ONE network (best.pt), with strict checks ----
import glob
model.eval()
assert not model.training, "model must be in eval mode"
assert sum(p.numel() for p in model.parameters()) <= 5_000_000, "model is over the 5M limit"
test_acc = acc(p_te, yte)
kp = predict(Xkg, tta=True).argmax(1).cpu().numpy()
sub = pd.DataFrame({"id": kaggle_ids.astype(int), "label": kp.astype(int)})

assert list(sub.columns) == ["id", "label"], "schema must be exactly id,label"
assert len(sub) == len(sample_sub) == len(Xkg), "one prediction per evaluation image"
assert (sub["id"].values == sample_sub["id"].values).all(), "ids/order must match sample_submission"
assert sub["id"].is_unique and not sub.isna().any().any(), "no duplicate or missing values"
assert sub["label"].between(0, 9).all() and pd.api.types.is_integer_dtype(sub["label"]), "labels must be integers 0-9"

stamp = time.strftime("%Y%m%d_%H%M")
drive_sub = os.path.join(RUN_DIR, f"submission_{stamp}_val{b['val_acc']:.4f}_test{test_acc:.4f}.csv")
sub.to_csv(drive_sub, index=False)
local_sub = "/content/submission.csv"; sub.to_csv(local_sub, index=False)
chk = pd.read_csv(local_sub)
assert chk.equals(sub), "re-read check failed"
print("saved:", drive_sub, "\nand:", local_sub)
print("predicted class counts:", dict(zip(CLASSES, np.bincount(kp, minlength=10).tolist())))

# how different is this from earlier submissions? (tells you how big a public-score change must be to be real)
prev = sorted(glob.glob(os.path.join(ROOT, "runs", "*", "submission_*.csv")))
prev = [p for p in prev if os.path.dirname(p) != RUN_DIR]
for p in prev[-6:]:
    d = (pd.read_csv(p)["label"].values != kp).mean()
    print(f"  differs from {os.path.relpath(p, ROOT)[:70]:70s} on {d:5.1%} -> public change < ~{2*math.sqrt(max(d,1e-9)/1000):.3f} is noise")

files.download(local_sub)